[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# The scikit-learn Interface &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' answers and models, rebuilt here from the same made-up stations so that each
answer runs on its own.

In [2]:
y_train, y_valid, y_test = train["tmin"], valid["tmin"], test["tmin"]
xgboost_model = xgb.XGBRegressor(random_state=0).fit(train[FEATURES], y_train)
station_folds = GroupKFold(n_splits=5, shuffle=True, random_state=0)


def add_season(rows):
    return rows.assign(season=np.cos(2 * np.pi * (rows["day_of_year"] - 20) / 365))

**1.** Print the `n_estimators` and `learning_rate` that `get_params()` holds for `LGBMClassifier()` and for `XGBClassifier()`.

In [3]:
for model in (lgb.LGBMClassifier(), xgb.XGBClassifier()):
    params = model.get_params()
    print(type(model).__name__, params["n_estimators"], params["learning_rate"])

LGBMClassifier 100 0.1
XGBClassifier None None


The classifiers hold the regressors' defaults. LightGBM's constructor holds 100 trees and a rate of 0.1, and XGBoost's holds `None` for both.

**2.** Fit an `XGBRegressor` with `learning_rate=0.1`, LightGBM's default, keeping its other defaults, and print its validation loss beside the default `xgboost_model`'s.

In [4]:
slower = xgb.XGBRegressor(learning_rate=0.1, random_state=0).fit(train[FEATURES], y_train)
print("learning rate 0.1:", round(mean_squared_error(y_valid, slower.predict(valid[FEATURES])), 3))
print("default, 0.3:", round(mean_squared_error(y_valid, xgboost_model.predict(valid[FEATURES])), 3))

learning rate 0.1: 2.544
default, 0.3: 2.924


At 0.1 XGBoost's validation loss is 2.544, against 2.924 at its default of 0.3, and close to LightGBM's 2.553 in the worked examples with the same rate and number of trees, in one run that cannot rank the two.

**3.** Cross-validate `XGBRegressor(random_state=0)` by station with `GroupKFold(n_splits=5, shuffle=True, random_state=0)`, and print the five losses and their mean.

In [5]:
scores = cross_val_score(xgb.XGBRegressor(random_state=0), train[FEATURES], y_train, groups=train["station"],
                         cv=station_folds, scoring="neg_mean_squared_error")
print((-scores).round(3), "| mean", round(-scores.mean(), 3))

[3.203 2.767 2.932 2.874 3.587] | mean 3.073


XGBoost's mean over the folds of stations is 3.073, against LightGBM's 2.788 on the same folds in the worked examples. Its folds range from 2.767 to 3.587, and its loss is higher than LightGBM's on each of the five, with defaults that differ in learning rate, tree size and penalty.

**4.** Put `FunctionTransformer(add_season)` in front of an `XGBRegressor(random_state=0)` and cross-validate the pipeline by station, then compare its mean loss with the previous task's.

In [6]:
pipeline = make_pipeline(FunctionTransformer(add_season), xgb.XGBRegressor(random_state=0))
with_season = cross_val_score(pipeline, train[FEATURES], y_train, groups=train["station"],
                              cv=station_folds, scoring="neg_mean_squared_error")
print((-with_season).round(3), "| mean", round(-with_season.mean(), 3))

[3.098 2.813 2.941 2.756 3.675] | mean 3.056


With the season the mean is 3.056, against 3.073 without it. The loss is lower on two of the five folds and higher on three, so these folds show no clear gain from the column.

**5.** Count how many test days each classifier gives a frost probability above 0.9, and what share of those days really had frost.

In [7]:
frost_train = (train["tmin"] < 0).astype(int)
frost_test = (test["tmin"] < 0).astype(int)
for model in (lgb.LGBMClassifier(random_state=0, verbose=-1), xgb.XGBClassifier(random_state=0)):
    model.fit(train[FEATURES], frost_train)
    sure = model.predict_proba(test[FEATURES])[:, 1] > 0.9
    print(type(model).__name__, int(sure.sum()), "days, share with frost", round(frost_test[sure].mean(), 3))

LGBMClassifier 1194 days, share with frost 0.986
XGBClassifier 1219 days, share with frost 0.981


LightGBM gives 1,194 test days a frost probability above 0.9, and 98.6% of them had frost. XGBoost gives 1,219 days, and 98.1% of them had frost.

---

&#8592; **Back to:** [The scikit-learn Interface](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/02-the-scikit-learn-interface.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
